In [1]:
from utils import Sql, train_val_test_split, get_sm_service_role_arn
from sklearn.metrics import mean_squared_error, r2_score, mean_absolute_error, root_mean_squared_error
import pandas as pd
import sagemaker, boto3, utils
import paths as p

region='us-east-1'
model_package_group_name='abalone'
model_version=1
data_bucket='omm-test-bucket'
project_path = 'models/abalone'
account='088461143167'
boto_session=boto3.Session(region_name=region)
sm_client = boto_session.client('sagemaker', region_name=region)
s3_resource = boto_session.resource('s3')

sagemaker.config INFO - Not applying SDK defaults from location: /etc/xdg/sagemaker/config.yaml
sagemaker.config INFO - Not applying SDK defaults from location: /home/ec2-user/.config/sagemaker/config.yaml


In [2]:
# DELETE ALL
model_package_group_name='abalone'
endpoint_name='abalone-endpoint'
endpoint_config_name='abalone-endpoint-config'


# Delete all models
models = sm_client.list_models()
for m in models['Models']:
    print(f"Deleting model: {m['ModelName']}")
    sm_client.delete_model(ModelName=m['ModelName'])

# Delete all model packages in the group
model_packages = sm_client.list_model_packages(
    ModelPackageGroupName=model_package_group_name
)
for mp in model_packages['ModelPackageSummaryList']:
    print(f"Deleting model package: {mp['ModelPackageArn']}")
    sm_client.delete_model_package(
        ModelPackageName=mp['ModelPackageArn']
    )

# Delete the model package group itself
sm_client.delete_model_package_group(
    ModelPackageGroupName=model_package_group_name
)
print("Deleted model package group: abalone")

# Delete endpoint first (must delete before config)
try:
    sm_client.delete_endpoint(EndpointName=endpoint_name)
    print("Deleted endpoint")
    # Wait for deletion
    waiter = sm_client.get_waiter('endpoint_deleted')
    waiter.wait(EndpointName=endpoint_name)
except sm_client.exceptions.ClientError:
    print("Endpoint does not exist")

# Delete endpoint config
try:
    sm_client.delete_endpoint_config(EndpointConfigName=endpoint_config_name)
    print("Deleted endpoint config")
except sm_client.exceptions.ClientError:
    print("Endpoint config does not exist")

# Delete all objects with a specific prefix (directory)
bucket = s3_resource.Bucket(data_bucket)
bucket.objects.filter(Prefix=f'{project_path}/data/batch-output/').delete()
bucket.objects.filter(Prefix=f'{project_path}/data/capture/').delete()
bucket.objects.filter(Prefix=f'{project_path}/data/ground-truth/').delete()
bucket.objects.filter(Prefix=f'{project_path}/data/monitors/').delete()

Deleting model: sagemaker-xgboost-2026-05-20-22-46-09-384
Deleting model: abalone-1
Deleting model package: arn:aws:sagemaker:us-east-1:088461143167:model-package/abalone/1
Deleted model package group: abalone
Endpoint does not exist
Deleted endpoint config


[]